# Jodhpur House Price Prediction
## 2. Model Training & Evaluation

This notebook trains multiple machine learning models using the cleaned dataset, evaluates their performance, and saves the best model (`price_prediction_model.pkl`) for use in the web application.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

### Feature Engineering
Prepare the data for the machine learning algorithms by extracting the target variable and encoding categorical features.

In [ ]:
# Load cleaned dataset
df = pd.read_csv('../data/jodhpur_housing_cleaned.csv')

# Separate target (Price) from features
X = df.drop('Price', axis=1)
y = df['Price']

# One-Hot Encode the Area_Name column
X = pd.get_dummies(X, columns=['Area_Name'], drop_first=True)

# Save the final feature names order for the web app
feature_names = list(X.columns)
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print(f"Number of features: {len(feature_names)}")
X.head()

### Train-Test Split & Scaling

In [ ]:
# Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit the standard scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
print("Scaler saved successfully.")

### Model Training

In [ ]:
# Define models to train
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = []
trained_models = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results)
results_df

### Save Best Model
Based on the R2 Score, we select the best model to deploy.

In [ ]:
best_model_name = results_df.loc[results_df['R2 Score'].idxmax()]['Model']
best_model = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")

# Save the best model
with open('price_prediction_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Model saved as 'price_prediction_model.pkl'")

### Visualize Model Performance

In [ ]:
y_pred_best = best_model.predict(X_test_scaled)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Actual vs Predicted Price ({best_model_name})')
plt.tight_layout()
plt.show()